### Tools

Models can request to call tools that perform task such as fetching data from the database, searching the web, or runnning code. Tools are pairing of:

1. A Schema, including the name of the tool, description, and/or argument defination(often a JSON format)
2. A function or coroutine to execute

In [1]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] =  os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:llama-3.1-8b-instant")

response= model.invoke("Write a story in 50 words")

In [2]:
response

AIMessage(content='A lone astronaut drifted through space, the stars twinkling like diamonds in the vast expanse. She recalled the words of her mentor: "The universe is full of mystery, but also full of wonder." A burst of stardust caught her attention, and she smiled, feeling the vastness of the unknown.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 42, 'total_tokens': 106, 'completion_time': 0.149489147, 'completion_tokens_details': None, 'prompt_time': 0.079821456, 'prompt_tokens_details': None, 'queue_time': 0.138550943, 'total_time': 0.229310603}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d63f7-4cba-7fb3-b3e4-6d2267f750de-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 64, 'total_tokens': 106})

In [3]:
print("Hello")

Hello


In [4]:
## Tools
from langchain.tools import tool

@tool # decorator
def get_weather(location: str) -> str:
    """Get the weather at any location """
    return f"The current weather is sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [5]:

response = model_with_tools.invoke("What's the weather in Mumbai?")
print(response)
# response.tool_calls

content='' additional_kwargs={'tool_calls': [{'id': 'mntjbbtmn', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.035189727, 'completion_tokens_details': None, 'prompt_time': 0.018958053, 'prompt_tokens_details': None, 'queue_time': 0.164135654, 'total_time': 0.05414778}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d63f7-4fde-7612-b452-d3d9f026e5e4-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'mntjbbtmn', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234}


### Tool execution loop

In [6]:
# step 1: Model generates tools call
messages  = [{"role": "user", "content": "What's the wearher in Mumbai?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
# messages

# step 2: Execution tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# step 3: Pass results back to the LLM model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

Note: The actual response may vary based on the current weather conditions. The above response is just a placeholder.


In [7]:
messages

[{'role': 'user', 'content': "What's the wearher in Mumbai?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9nbzatfsp', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.02740186, 'completion_tokens_details': None, 'prompt_time': 0.015503939, 'prompt_tokens_details': None, 'queue_time': 0.048376185, 'total_time': 0.042905799}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d63f7-5172-7332-8284-52886961a88c-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': '9nbzatfsp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}),
 ToolMessage(content='The curr